In [2]:
#!/usr/bin/env python3
"""
Gemini test accuracy evaluation.
Calculates accuracy from match tags in test answer comparison.
"""

import json
from tqdm import tqdm

# Load Gemini test accuracy results
json_file = '/home/as5606/projects/Hulu-Med/gemini/gemini_test_accuracy_comparison.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        gemini_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(gemini_data)} samples\n")

# Count matches and no matches
match_count = 0
no_match_count = 0

for item in tqdm(gemini_data, desc="Evaluating answers"):
    if item.get('match') == 'MATCH':
        match_count += 1
    elif item.get('match') == 'NO_MATCH':
        no_match_count += 1

# Calculate accuracy
total = match_count + no_match_count
accuracy = (match_count / total * 100) if total > 0 else 0

# Print results
print(f"\n{'='*60}")
print(f"GEMINI TEST ACCURACY")
print(f"{'='*60}")
print(f"Total questions: {total}")
print(f"Matched: {match_count}")
print(f"No Match: {no_match_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")


Loading /home/as5606/projects/Hulu-Med/gemini/gemini_test_accuracy_comparison.json...
Loaded 7650 samples



Evaluating answers: 100%|██████████| 7650/7650 [00:00<00:00, 2463827.51it/s]


GEMINI TEST ACCURACY
Total questions: 7650
Matched: 3827
No Match: 3823
Accuracy: 50.03%


In [12]:
#!/usr/bin/env python3
"""
Gemini grounded questions accuracy evaluation.
Calculates accuracy from match tags in grounded answer comparison.
"""

import json
from tqdm import tqdm

# Load Gemini grounded accuracy results
json_file = '/home/as5606/projects/Hulu-Med/gemini/gemini_grounded_accuracy.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        grounded_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(grounded_data)} samples\n")

# Count matches
total_qa = 0
match_count = 0
no_match_count = 0

for item in tqdm(grounded_data, desc="Evaluating grounded answers"):
    for qa in item.get('grounded_qa_with_matches', []):
        total_qa += 1
        if qa['match'] == 'MATCH':
            match_count += 1
        elif qa['match'] == 'NO_MATCH':
            no_match_count += 1

# Calculate accuracy
accuracy = (match_count / total_qa * 100) if total_qa > 0 else 0

# Print results
print(f"\n{'='*60}")
print(f"GEMINI GROUNDED QUESTIONS ACCURACY")
print(f"{'='*60}")
print(f"Total grounded QA pairs: {total_qa}")
print(f"Matched: {match_count}")
print(f"No Match: {no_match_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")


Loading /home/as5606/projects/Hulu-Med/gemini/gemini_grounded_accuracy.json...
Loaded 1397 samples



Evaluating grounded answers: 100%|██████████| 1397/1397 [00:00<00:00, 670110.10it/s]


GEMINI GROUNDED QUESTIONS ACCURACY
Total grounded QA pairs: 5579
Matched: 3345
No Match: 2234
Accuracy: 59.96%


In [4]:
#!/usr/bin/env python3
"""
Gemini semantic variations accuracy evaluation for "how many" questions.
Adds match tags to "how many" questions and overwrites the original JSON file.
"""

import json
import os
from tqdm import tqdm
from openai import OpenAI

client = OpenAI(api_key="os.environ.get("OPENAI_API_KEY")")

# System prompt for "how many" questions
SYSTEM_PROMPT = """\
You are an answer-matching evaluator for surgical VQA counting questions.

Your task is to compare two answers about quantity and decide whether they express the same number.

Return only one of the following labels:

MATCH
NO_MATCH

Guidelines:
- Mark MATCH if both answers refer to the SAME QUANTITY, regardless of wording
- Mark MATCH for number variations: "2" matches "Two" or "Two tools are operating"
- Mark MATCH for variations: "1" matches "One" or "One tool" or "There is one tool"
- Mark MATCH for written forms: "3" matches "three" or "There are three tools"
- Mark MATCH even if one answer is more verbose: "2" matches "Two surgical instruments are visible"
- Mark MATCH for number words: "four" matches "4" matches "There are 4 tools operating"
- Mark NO_MATCH if the numbers differ: "2" vs "3"
- Mark NO_MATCH if one says "no tools" or "0" and the other says there are tools
- Ignore capitalization, punctuation, and extra wording

Input format:
Answer 1: <first answer>
Answer 2: <second answer>

Output format:
MATCH or NO_MATCH
"""

def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers and return whether they match in meaning."""
    user_text = f"Answer 1: {answer1}\nAnswer 2: {answer2}"
    
    try:
        response = client.responses.create(
            model="gpt-5.4-nano",
            instructions=SYSTEM_PROMPT,
            input=user_text,
        )
        
        result = response.output_text.strip()
        
        # Extract MATCH or NO_MATCH from response
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"
    
    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None

# Load semantic variations answers
json_file = '/home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        semantic_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(semantic_data)} samples\n")

# Add match field for "how many" questions only
how_many_count = 0
matched_count = 0

for item in tqdm(semantic_data, desc="Adding match field for 'how many' questions"):
    original_question = item['original_question'].lower()
    
    # Only evaluate "how many" questions
    if "how many" in original_question:
        how_many_count += 1
        original_answer = item['original_answer']
        
        # Add match field to each semantic variation
        for variation in item.get('semantic_variations_with_gemini_answers', []):
            gemini_answer = variation['gemini_answer']
            
            # Compare answers
            match = compare_answers(original_answer, gemini_answer)
            variation['match'] = match
            
            if match == 'MATCH':
                matched_count += 1

# Overwrite original file with updated data
with open(json_file, 'w') as f:
    json.dump(semantic_data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated and saved {len(semantic_data)} items to {json_file}")

# Print summary
print(f"\n{'='*60}")
print(f"Gemini 'HOW MANY' SEMANTIC VARIATIONS EVALUATION")
print(f"{'='*60}")
print(f"Total 'how many' questions: {how_many_count}")
print(f"Matched variations: {matched_count}")
if how_many_count > 0:
    # Count total variations for "how many" questions
    total_variations = 0
    for item in semantic_data:
        if "how many" in item['original_question'].lower():
            total_variations += len(item.get('semantic_variations_with_gemini_answers', []))
    
    accuracy = (matched_count / total_variations * 100) if total_variations > 0 else 0
    print(f"Total variations: {total_variations}")
    print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")


Loading /home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json...
Loaded 7646 samples



Adding match field for 'how many' questions: 100%|██████████| 7646/7646 [44:32<00:00,  2.86it/s]  


✓ Updated and saved 7646 items to /home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json

Gemini 'HOW MANY' SEMANTIC VARIATIONS EVALUATION
Total 'how many' questions: 946
Matched variations: 3557
Total variations: 4730
Accuracy: 75.20%


In [5]:
#!/usr/bin/env python3
"""
Gemini semantic variations consistency score evaluation.
Calculates Consistency Score (CS) based on:
- "is used"/"is not used" questions: check for "yes"/"no"
- "how many" questions: check match tag
"""

import json
from tqdm import tqdm

# Load Gemini semantic variations answers
json_file = '/home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        semantic_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(semantic_data)} samples\n")

# Calculate Consistency Score
cs_sum = 0  # Sum of per-item consistency scores
total_items = 0

for item in tqdm(semantic_data, desc="Evaluating consistency"):
    original_question = item['original_question'].lower()
    original_answer = item['original_answer'].lower()
    variations = item.get('semantic_variations_with_gemini_answers', [])
    
    if len(variations) == 0:
        continue
    
    total_items += 1
    matches = 0
    K = len(variations)
    
    for variation in variations:
        gemini_answer = variation['gemini_answer'].lower()
        is_correct = False
        
        # Check for "is used" questions
        if "is used" in original_answer and "is not used" not in original_answer:
            if "yes" in gemini_answer:
                is_correct = True
        
        # Check for "is not used" questions
        elif "is not used" in original_answer:
            if "no" in gemini_answer:
                is_correct = True
        
        # Check for "how many" questions
        elif "how many" in original_question:
            if variation.get('match') == 'MATCH':
                is_correct = True
        
        if is_correct:
            matches += 1
    
    # Calculate per-item consistency: (1/K) * Σ matches
    cs_per_item = matches / K
    cs_sum += cs_per_item

# Calculate final Consistency Score
CS = cs_sum / total_items if total_items > 0 else 0

# Print results
print(f"\n{'='*70}")
print(f"GEMINI SEMANTIC VARIATIONS CONSISTENCY SCORE")
print(f"{'='*70}\n")

print(f"Consistency Score (CS): {CS:.4f}")
print(f"  Interpretation: {CS*100:.2f}% of semantic variations answered correctly")
print(f"  Perfect CS: 1.0 (all variations answered consistently)\n")

print(f"{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total questions evaluated: {total_items}")
if CS >= 0.8:
    print(f"Status: ✅ EXCELLENT consistency (CS >= 0.8)")
elif CS >= 0.7:
    print(f"Status: ✅ GOOD consistency (CS >= 0.7)")
elif CS >= 0.6:
    print(f"Status: ⚠️  MODERATE consistency (CS >= 0.6)")
else:
    print(f"Status: ❌ POOR consistency (needs improvement)")

print(f"\nTarget: CS = 1.0 (100% correct)")
print(f"Current: CS = {CS:.4f} ({CS*100:.2f}%)")
print(f"{'='*70}")


Loading /home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json...
Loaded 7646 samples



Evaluating consistency: 100%|██████████| 7646/7646 [00:00<00:00, 298000.75it/s]


GEMINI SEMANTIC VARIATIONS CONSISTENCY SCORE

Consistency Score (CS): 0.5270
  Interpretation: 52.70% of semantic variations answered correctly
  Perfect CS: 1.0 (all variations answered consistently)

SUMMARY
Total questions evaluated: 7646
Status: ❌ POOR consistency (needs improvement)

Target: CS = 1.0 (100% correct)
Current: CS = 0.5270 (52.70%)


In [6]:
#!/usr/bin/env python3
"""
Calculate Answer Variance Score (AVS) for Gemini semantic variations.
AVS measures consistency of answers across semantic variations.
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/gemini/gemini_semantic_variations_answers.json', 'r') as f:
    data = json.load(f)

total_variance = 0

for item in tqdm(data):
    # Collect all answers for this item's variations
    answers = []

    for variation in item.get('semantic_variations_with_gemini_answers', []):
        gemini_answer = variation['gemini_answer'].lower().strip()
        answers.append(gemini_answer)

    # Count distinct answers
    distinct_answers = len(set(answers))
    total_variance += distinct_answers

# Calculate AVS
avs = total_variance / len(data)
print(f"\nAnswer Variance Score (AVS): {avs:.4f}")
print(f"Interpretation:")
print(f"  - AVS = 1: Perfect consistency (same answer for all rephrasings)")
print(f"  - AVS > 1: Instability (different answers for same question)")


100%|██████████| 7646/7646 [00:00<00:00, 404970.94it/s]


Answer Variance Score (AVS): 3.0955
Interpretation:
  - AVS = 1: Perfect consistency (same answer for all rephrasings)
  - AVS > 1: Instability (different answers for same question)


In [8]:
#!/usr/bin/env python3
"""
Calculate Abstention Rate (AR) for Gemini on hallucinatory (unanswerable) questions.
AR measures the proportion of unanswerable questions on which the model correctly abstains.
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/gemini/gemini_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

total_questions = 0
abstention_count = 0

for item in tqdm(data, desc="Checking abstention rate"):
    new_questions = item.get('new_questions_with_gemini_answers', [])
    
    for question in new_questions:
        gemini_answer = question.get('gemini_answer')
        
        # Skip if answer is None (missing/failed generation)
        if gemini_answer is None:
            continue
        
        gemini_answer = gemini_answer.lower().strip()
        total_questions += 1
        
        # Check if answer is "i don't know" (with or without punctuation)
        if "i don't know" in gemini_answer or "i dont know" in gemini_answer:
            abstention_count += 1

# Calculate Abstention Rate
ar = abstention_count / total_questions if total_questions > 0 else 0

print(f"\n{'='*70}")
print(f"GEMINI HALLUCINATORY QUESTIONS ABSTENTION RATE")
print(f"{'='*70}\n")

print(f"Total unanswerable questions: {total_questions}")
print(f"Abstentions (I don't know): {abstention_count}")
print(f"Abstention Rate (AR): {ar:.4f}")
print(f"  Percentage: {ar*100:.2f}%\n")

print(f"Interpretation:")
print(f"  - AR = 1.0: Perfect (all unanswerable questions abstained)")
print(f"  - AR = 0.0: Poor (no abstentions, likely hallucinating)")
print(f"  - Higher AR = Better model behavior on unanswerable questions")
print(f"\n{'='*70}")


Checking abstention rate: 100%|██████████| 7648/7648 [00:00<00:00, 675654.25it/s]


GEMINI HALLUCINATORY QUESTIONS ABSTENTION RATE

Total unanswerable questions: 15293
Abstentions (I don't know): 6115
Abstention Rate (AR): 0.3999
  Percentage: 39.99%

Interpretation:
  - AR = 1.0: Perfect (all unanswerable questions abstained)
  - AR = 0.0: Poor (no abstentions, likely hallucinating)
  - Higher AR = Better model behavior on unanswerable questions



In [9]:
#!/usr/bin/env python3
"""
Calculate False Confidence Rate (FCR) for Gemini on answerable questions.
FCR measures the proportion of answerable questions on which the model incorrectly abstains.
Higher FCR = worse (model being overly cautious on answerable questions).
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/gemini/gemini_test_prompt_answers.json', 'r') as f:
    data = json.load(f)

total_questions = 0
abstain_count = 0

for item in tqdm(data, desc="Checking false confidence rate"):
    gemini_answer = item.get('gemini_answer')
    
    # Skip if answer is None
    if gemini_answer is None:
        continue
    
    gemini_answer = gemini_answer.lower().strip()
    total_questions += 1
    
    # Check if answer is "i don't know" (with or without punctuation)
    if "i don't know" in gemini_answer or "i dont know" in gemini_answer:
        abstain_count += 1

# Calculate False Confidence Rate
fcr = abstain_count / total_questions if total_questions > 0 else 0

print(f"\n{'='*70}")
print(f"GEMINI ANSWERABLE QUESTIONS - FALSE CONFIDENCE RATE")
print(f"{'='*70}\n")

print(f"Total answerable questions: {total_questions}")
print(f"Incorrectly abstained (I don't know): {abstain_count}")
print(f"False Confidence Rate (FCR): {fcr:.4f}")
print(f"  Percentage: {fcr*100:.2f}%\n")

print(f"Interpretation:")
print(f"  - FCR = 0.0: Perfect (answered all questions that should be answered)")
print(f"  - FCR = 1.0: Worst (abstained on all answerable questions)")
print(f"  - Higher FCR = Worse model behavior (overly cautious)")
print(f"\n{'='*70}")


Checking false confidence rate: 100%|██████████| 7652/7652 [00:00<00:00, 1458855.19it/s]


GEMINI ANSWERABLE QUESTIONS - FALSE CONFIDENCE RATE

Total answerable questions: 7652
Incorrectly abstained (I don't know): 3177
False Confidence Rate (FCR): 0.4152
  Percentage: 41.52%

Interpretation:
  - FCR = 0.0: Perfect (answered all questions that should be answered)
  - FCR = 1.0: Worst (abstained on all answerable questions)
  - Higher FCR = Worse model behavior (overly cautious)



In [11]:
#!/usr/bin/env python3
"""
Analyze the effect of prompts on Gemini answers.
Shows how many correct answers change to "I don't know" when prompted.
"""

import json
from tqdm import tqdm

# Load both files
with open('/home/as5606/projects/Hulu-Med/gemini/gemini_test_accuracy_comparison.json', 'r') as f:
    original_answers = json.load(f)

with open('/home/as5606/projects/Hulu-Med/gemini/gemini_test_prompt_answers.json', 'r') as f:
    prompt_answers = json.load(f)

# Create mapping of ID to prompt answer for quick lookup
prompt_map = {item['id']: item for item in prompt_answers}

# Count metrics
original_correct = 0
changed_to_idk = 0
total_questions = 0

for item in tqdm(original_answers, desc="Analyzing prompt effect"):
    item_id = item['id']
    
    # Only count items with match field
    if 'match' not in item:
        continue
    
    total_questions += 1
    
    # Check if original answer was correct
    if item.get('match') == 'MATCH':
        original_correct += 1
        
        # Check if same question in prompted version changed to "I don't know"
        if item_id in prompt_map:
            prompt_answer = prompt_map[item_id].get('gemini_answer', '').lower().strip()
            
            if "i don't know" in prompt_answer or "i dont know" in prompt_answer:
                changed_to_idk += 1

# Calculate accuracies
original_accuracy = (original_correct / total_questions) * 100 if total_questions > 0 else 0
new_accuracy = ((original_correct - changed_to_idk) / total_questions) * 100 if total_questions > 0 else 0
accuracy_drop = original_accuracy - new_accuracy

print(f"\n{'='*70}")
print(f"GEMINI PROMPT EFFECT ANALYSIS")
print(f"{'='*70}\n")

print(f"Original accuracy: {original_accuracy:.2f}%")
print(f"Originally correct answers: {original_correct}/{total_questions}")
print(f"\nAnswers changed to 'I don't know' due to prompt: {changed_to_idk}")
print(f"New accuracy (after prompt): {new_accuracy:.2f}%")
print(f"Accuracy drop: {accuracy_drop:.2f}%\n")

print(f"{'='*70}")
print(f"Interpretation:")
print(f"  - Negative accuracy drop = Prompt made model more cautious")
print(f"  - Larger drop = Prompt significantly degrades model reliability")
print(f"{'='*70}")


Analyzing prompt effect: 100%|██████████| 7650/7650 [00:00<00:00, 1238284.41it/s]


GEMINI PROMPT EFFECT ANALYSIS

Original accuracy: 50.03%
Originally correct answers: 3827/7650

Answers changed to 'I don't know' due to prompt: 1545
New accuracy (after prompt): 29.83%
Accuracy drop: 20.20%

Interpretation:
  - Negative accuracy drop = Prompt made model more cautious
  - Larger drop = Prompt significantly degrades model reliability
